# Module 07-1: LLM 비용 계산기 (솔루션)

## 학습 목표
- LLM API의 토큰 기반 과금 구조를 이해할 수 있다
- `estimate_cost()` 함수로 모델별 비용을 계산할 수 있다
- Haiku vs Sonnet의 비용 절감율을 계산할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/07-llm-call-optimization.md` 섹션 1.1

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

print("환경 설정 완료")

## 실습 1: estimate_cost() 함수 구현 (솔루션)

In [ ]:
# 솔루션
def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """LLM 호출 비용을 추정한다 (USD).
    
    Args:
        model: 모델 이름 ('claude-haiku', 'claude-sonnet', 'claude-opus', 'gpt-4o-mini', 'gpt-4o')
        input_tokens: 입력 토큰 수
        output_tokens: 출력 토큰 수
    
    Returns:
        추정 비용 (USD), 소수점 6자리 반올림
    """
    pricing = {
        "claude-haiku":  {"input": 1.00, "output": 5.00},
        "claude-sonnet": {"input": 3.00, "output": 15.00},
        "claude-opus":   {"input": 15.00, "output": 75.00},
        "gpt-4o-mini":   {"input": 0.15, "output": 0.60},
        "gpt-4o":        {"input": 2.50, "output": 10.00},
    }
    price = pricing.get(model, pricing["claude-sonnet"])
    cost = (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000
    return round(cost, 6)


sonnet_cost = estimate_cost("claude-sonnet", input_tokens=2000, output_tokens=500)
haiku_cost = estimate_cost("claude-haiku", input_tokens=2000, output_tokens=500)

print(f"Sonnet 비용: ${sonnet_cost}")
print(f"Haiku 비용: ${haiku_cost}")

In [ ]:
# 검증 셀
assert estimate_cost is not None and estimate_cost("claude-sonnet", 1000, 100) is not None, \
    "estimate_cost 함수를 구현하세요."

# Sonnet 비용: (2000 * 3.00 + 500 * 15.00) / 1_000_000 = (6000 + 7500) / 1M = 0.0135
expected_sonnet = 0.0135
actual_sonnet = estimate_cost("claude-sonnet", 2000, 500)
assert abs(actual_sonnet - expected_sonnet) < 0.0001, \
    f"Sonnet 비용이 ${expected_sonnet}이어야 합니다. 현재: ${actual_sonnet}"

# Haiku 비용: (2000 * 1.00 + 500 * 5.00) / 1_000_000 = (2000 + 2500) / 1M = 0.0045
expected_haiku = 0.0045
actual_haiku = estimate_cost("claude-haiku", 2000, 500)
assert abs(actual_haiku - expected_haiku) < 0.0001, \
    f"Haiku 비용이 ${expected_haiku}이어야 합니다. 현재: ${actual_haiku}"

print("실습 1 완료!")

## 실습 2: 모델별 비용 비교 표 출력

In [ ]:
# 2-1. 모든 모델 비용 계산 및 출력
models = ["claude-haiku", "claude-sonnet", "claude-opus", "gpt-4o-mini", "gpt-4o"]
input_tokens = 2000
output_tokens = 500

print(f"=== 모델별 비용 비교 (입력 {input_tokens}토큰, 출력 {output_tokens}토큰) ===")
print(f"{'모델':<20} {'비용 (USD)':<15} {'Sonnet 대비 비율':<15}")
print("-" * 55)

sonnet_cost = estimate_cost("claude-sonnet", input_tokens, output_tokens)

costs = {}
for model in models:
    cost = estimate_cost(model, input_tokens, output_tokens)
    costs[model] = cost
    ratio = cost / sonnet_cost
    print(f"{model:<20} ${cost:<14.6f} {ratio:.2f}x")

print("-" * 55)
cheapest = min(costs, key=costs.get)
most_expensive = max(costs, key=costs.get)
print(f"\n가장 저렴: {cheapest} (${costs[cheapest]:.6f})")
print(f"가장 비쌈: {most_expensive} (${costs[most_expensive]:.6f})")

In [ ]:
# 검증 셀
assert len(costs) == 5, "5개 모델의 비용이 계산되어야 합니다."
assert costs["claude-haiku"] < costs["claude-sonnet"], "Haiku가 Sonnet보다 저렴해야 합니다."
assert costs["claude-sonnet"] < costs["claude-opus"], "Sonnet이 Opus보다 저렴해야 합니다."
print("실습 2 완료!")

## 실습 3: Haiku vs Sonnet 절감율 계산 (솔루션)

In [ ]:
# 솔루션
daily_requests = 1000
avg_input_tokens = 2000
avg_output_tokens = 500
days = 30

sonnet_per_call = estimate_cost("claude-sonnet", avg_input_tokens, avg_output_tokens)
haiku_per_call = estimate_cost("claude-haiku", avg_input_tokens, avg_output_tokens)

total_sonnet = sonnet_per_call * daily_requests * days
total_haiku = haiku_per_call * daily_requests * days
savings = total_sonnet - total_haiku
savings_rate = (savings / total_sonnet) * 100

print("=== Haiku vs Sonnet 월별 비용 비교 ===")
print(f"일별 요청 수: {daily_requests:,}건")
print(f"기간: {days}일")
print(f"\nSonnet 월별 비용: ${total_sonnet:,.2f}")
print(f"Haiku 월별 비용:  ${total_haiku:,.2f}")
print(f"절감 금액: ${savings:,.2f}")
print(f"절감율: {savings_rate:.1f}%")

In [ ]:
# 검증 셀
assert total_sonnet is not None, "total_sonnet을 계산하세요."
assert total_haiku is not None, "total_haiku를 계산하세요."
assert savings is not None, "savings를 계산하세요."
assert savings_rate is not None, "savings_rate를 계산하세요."

# Sonnet: 0.0135 × 1000 × 30 = 405
assert abs(total_sonnet - 405.0) < 1.0, f"Sonnet 월별 비용이 약 $405이어야 합니다. 현재: ${total_sonnet:.2f}"

# Haiku: 0.0045 × 1000 × 30 = 135
assert abs(total_haiku - 135.0) < 1.0, f"Haiku 월별 비용이 약 $135이어야 합니다. 현재: ${total_haiku:.2f}"

# 절감율: (405 - 135) / 405 = 0.667 = 66.7%
assert abs(savings_rate - 66.67) < 1.0, f"절감율이 약 66.7%이어야 합니다. 현재: {savings_rate:.1f}%"

print("실습 3 완료!")

## 정리

1. **토큰 과금**: 입력 토큰과 출력 토큰이 별도로 과금되며, 출력이 3~5배 비쌈
2. **비용 계산 공식**: `(입력_토큰 × 입력_가격 + 출력_토큰 × 출력_가격) / 1,000,000`
3. **모델 선택의 중요성**: Haiku vs Sonnet은 동일 작업에서 약 **67% 비용 절감** 가능
4. **운영 규모**: 하루 1,000건 기준, 월 $270 절감 효과